# Module 2 Assessment
Welcome to the assessment for Module 2: Mapping for Planning. In this assessment, you will be generating an occupancy grid using lidar scanner measurements from a moving vehicle in an unknown environment. You will use the inverse scanner measurement model developed in the lessons to map these measurements into occupancy probabilities, and then perform iterative logodds updates to an occupancy grid belief map. After the car has gathered enough data, your occupancy grid should converge to the true map.

In this assessment, you will:
* Gather range measurements of a moving car's surroundings using a lidar scanning function.
* Extract occupancy information from the range measurements using an inverse scanner model.
* Perform logodds updates on an occupancy grids based on incoming measurements.
* Iteratively construct a probabilistic occupancy grid from those log odds updates.

For most exercises, you are provided with a suggested outline. You are encouraged to diverge from the outline if you think there is a better, more efficient way to solve a problem.
Launch the Jupyter Notebook to begin!

In [2]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.animation as anim
from IPython.display import HTML
import carla
import cv2 # برای نمایش بصری نقشه
m= None

In this notebook, you will generate an occupancy grid based off of multiple simulated lidar scans. The inverse scanner model will be given to you, in the `inverse_scanner()` function. It returns a matrix of measured occupancy probability values based on the lidar scan model discussed in the video lectures. The `get_ranges()` function actually returns the scanned ranges value for a given vehicle position and scanner bearing. These two functions are given below. Make sure you understand what they are doing, as you will need to use them later in the notebook.

In [3]:
# Generates range measurements for a laser scanner based on a map, vehicle position,
# and sensor parameters.
# Uses the ray tracing algorithm.
def get_ranges(true_map, X, meas_phi, rmax): # X is the location of the robot
    (M, N) = np.shape(true_map)
    x = X[0]
    y = X[1]
    theta = X[2]
    meas_r = rmax * np.ones(meas_phi.shape)
    
    # Iterate for each measurement bearing.
    for i in range(len(meas_phi)):
        # Iterate over each unit step up to and including rmax.
        for r in range(1, rmax+1):
            # Determine the coordinates of the cell.
            xi = int(round(x + r * math.cos(theta + meas_phi[i])))
            yi = int(round(y + r * math.sin(theta + meas_phi[i])))
            
            # If not in the map, set measurement there and stop going further.
            if (xi <= 0 or xi >= M-1 or yi <= 0 or yi >= N-1):
                meas_r[i] = r
                break
            # If in the map, but hitting an obstacle, set the measurement range
            # and stop ray tracing.
            elif true_map[int(round(xi)), int(round(yi))] == 1:
                meas_r[i] = r
                break
                
    return meas_r

In [4]:
# Calculates the inverse measurement model for a laser scanner.
# It identifies three regions. The first where no information is available occurs
# outside of the scanning arc. The second where objects are likely to exist, at the
# end of the range measurement within the arc. The third are where objects are unlikely
# to exist, within the arc but with less distance than the range measurement.
def inverse_scanner(num_rows, num_cols, x, y, theta, meas_phi, meas_r, rmax, alpha, beta):
    m = np.zeros((M, N))
    for i in range(num_rows):
        for j in range(num_cols):
            # Find range and bearing relative to the input state (x, y, theta).
            r = math.sqrt((i - x)**2 + (j - y)**2)
            phi = (math.atan2(j - y, i - x) - theta + math.pi) % (2 * math.pi) - math.pi
            
            # Find the range measurement associated with the relative bearing.
            k = np.argmin(np.abs(np.subtract(phi, meas_phi))) #This line finds which sensor beam angle in meas_phi is closest to the bearing angle phi
            
            # If the range is greater than the maximum sensor range, or behind our range
            # measurement, or is outside of the field of view of the sensor, then no
            # no information is available.
            if (r > min(rmax, meas_r[k] + alpha / 2.0)) or (abs(phi - meas_phi[k]) > beta / 2.0):
                m[i, j] = 0.5
            
            # If the range measurement lied within this cell, it is likely to be an object.
            elif (meas_r[k] < rmax) and (abs(r - meas_r[k]) < alpha / 2.0):
                m[i, j] = 0.7
            
            # If the cell is in front of the range measurement, it is likely to be empty.
            elif r < meas_r[k]:
                m[i, j] = 0.3
                
    return m
                            

In [5]:
def world_to_grid(x_world, y_world, M, N, resolution, origin_x, origin_y):
    """تبدیل مختصات جهانی CARLA (متر) به مختصات شبکه (پیکسل i, j)."""
    # توجه: CARLA x/y را به عنوان طول/عرض می بیند. ما x را به row (i) و y را به col (j) می نگاریم.
    i = int(round(origin_x - y_world / resolution)) # نگاشت Y CARLA (شمال/جنوب) به ردیف
    j = int(round(origin_y + x_world / resolution)) # نگاشت X CARLA (شرق/غرب) به ستون
    
    if 0 <= i < M and 0 <= j < N:
        return i, j
    return None, None

In [6]:
def grid_to_world(i, j, resolution, origin_x, origin_y):
    """تبدیل مختصات شبکه (پیکسل i, j) به مختصات جهانی CARLA (متر x, y)."""
    y_world = -(i - origin_x) * resolution
    x_world = (j - origin_y) * resolution
    return x_world, y_world

In [7]:
def lidar_callback(data):
    """
    تابع Callback برای پردازش داده‌های خام LiDAR.
    """
    global lidar_points
    points = np.frombuffer(data.raw_data, dtype=np.float32)
    
    # تعداد کامل نقاط (هر نقطه 4 فیلد: x, y, z, intensity)
    n_points = len(points) // 4
    
    # فقط بخش کامل داده‌ها رو reshape می‌کنیم
    lidar_points = np.reshape(points[:n_points*4], (n_points, 4))
# --- تابع جدید: Callback برای دوربین ---
def camera_callback(data):
    """
    تابع Callback برای پردازش داده های خام دوربین RGB.
    """
    global camera_image
    # تبدیل داده خام دوربین (BGRA) به آرایه NumPy
    img_data = np.frombuffer(data.raw_data, dtype=np.uint8)
    # تغییر شکل به ابعاد (ارتفاع، عرض، کانال‌ها)
    img = np.reshape(img_data, (data.height, data.width, 4))
    # حذف کانال آلفا (A) و تبدیل از BGRA به RGB یا BGR برای OpenCV
    camera_image = img[:, :, :3]
    # CARLA به صورت BGRA خروجی می‌دهد. OpenCV از BGR استفاده می‌کند.
    # camera_image = camera_image[:, :, ::-1] # اگر لازم بود از BGR به RGB تغییر دهید

In [8]:
def lidar_points_to_ranges(lidar_points, meas_phi, rmax):
    """
    نقاط خام لایدار (در فریم سنسور) را به برد متناظر با meas_phi تبدیل می‌کند.
    """
    if lidar_points is None:
        return rmax * np.ones_like(meas_phi)

    # x جلو، y چپ
    xs = lidar_points[:, 0]
    ys = lidar_points[:, 1]

    ranges = np.sqrt(xs**2 + ys**2)
    phis   = np.arctan2(ys, xs)

    meas_r = rmax * np.ones_like(meas_phi)

    for i in range(len(meas_phi)):
        diff = phis - meas_phi[i]
        # ترفند برای اختلاف زاویه در محدوده [-pi, pi]
        diff = np.arctan2(np.sin(diff), np.cos(diff))
        
        # آستانه: یافتن نقاطی که در محدوده زاویه‌ای بتا قرار دارند.
        indices = np.where(np.abs(diff) < beta / 2.0)[0]   

        if len(indices) > 0:
            # از بین نقاطی که در محدوده زاویه هستند، کمترین برد را انتخاب کن
            meas_r[i] = np.min(ranges[indices])

    return meas_r

In [9]:
def inverse_scanner_carla(M, N, x_carla, y_carla, theta_carla, meas_phi, meas_r, rmax, alpha, beta):
    """
    مدل سنسور معکوس، سازگار شده با مختصات و منطق ارائه شده شما.
    """
    m = 0.5 * np.ones((M, N))
    
    for i in range(M):
        for j in range(N):
            # مختصات جهانی CARLA مرکز سلول (i, j)
            x_world, y_world = grid_to_world(i, j, GRID_RESOLUTION, GRID_ORIGIN_X, GRID_ORIGIN_Y)

            # برد و زاویه سلول نسبت به خودرو
            r = np.sqrt((x_world - x_carla)**2 + (y_world - y_carla)**2)
            phi = np.arctan2(y_world - y_carla, x_world - x_carla)

            # زاویه در فریم خودرو
            phi_car = phi - theta_carla
            phi_car = np.arctan2(np.sin(phi_car), np.cos(phi_car)) 

            # پیدا کردن نزدیک‌ترین پرتو مدل
            k = np.argmin(np.abs(np.subtract(phi_car, meas_phi))) 
            r_k = meas_r[k]
            phi_k = meas_phi[k]
            
            # --- منطق احتمال اشغال بر اساس تابع شما ---
            
            # 1. بدون اطلاعات (احتمال 0.5)
            if (r > min(rmax, r_k + alpha / 2.0)) or (abs(phi_car - phi_k) > beta / 2.0):
                m[i, j] = 0.5
            
            # 2. اشغال شده (احتمال 0.7)
            elif (r_k < rmax) and (abs(r - r_k) < alpha / 2.0):
                m[i, j] = 0.7
            
            # 3. آزاد (احتمال 0.3)
            elif r < r_k:
                m[i, j] = 0.3
            
    return m

In the following code block, we initialize the required variables for our simulation. This includes the initial state as well as the set of control actions for the car. We also set the rate of rotation of our lidar scan. The obstacles of the true map are represented by 1's in the true map, 0's represent free space. Each cell in the belief map `m` is initialized to 0.5 as our prior probability of occupancy, and from that belief map we compute our logodds occupancy grid `L`.

In [10]:
M = 200
N = 200
#meas_phi = np.arange(-0.4, 0.4, 0.05)
meas_phi = np.linspace(-math.pi, math.pi, 360, endpoint=False)
rmax = 30 # Max beam range.
alpha = 1 # Width of an obstacle (distance about measurement to fill in).
beta = 0.05 # Angular width of a beam.
# Log Odds اولیه
m_prior = np.multiply(0.5, np.ones((M, N)))
L0 = np.log(np.divide(m_prior, np.subtract(1, m_prior)))
L = L0 # ماتریس Log Odds اصلی

# وضوح نقشه: هر سلول چه مسافتی را پوشش می‌دهد (متر بر پیکسل)
GRID_RESOLUTION = 0.5 # فرض 0.5 متر بر پیکسل
GRID_ORIGIN_X, GRID_ORIGIN_Y = M // 2, N // 2 # مرکز نقشه در (M/2, N/2)

# --- 2. متغیرهای جهانی CARLA و LiDAR ---
lidar_points = None
camera_image = None # متغیر جدید برای ذخیره فریم دوربین
vehicle = None
lidar_sensor = None
camera_sensor = None # سنسور دوربین


Here is where you will enter your code. Your task is to complete the main simulation loop. After each step of robot motion, you are required to gather range data from your lidar scan, and then apply the inverse scanner model to map these to a measured occupancy belief map. From this, you will then perform a logodds update on your logodds occupancy grid, and update our belief map accordingly. As the car traverses through the environment, the occupancy grid belief map should move closer and closer to the true map. At the code block after the end of the loop, the code will output some values which will be used for grading your assignment. Make sure to copy down these values and save them in a .txt file for when your visualization looks correct. Good luck!

In [14]:
def run_mapping_simulation():
    global L, vehicle, lidar_sensor, camera_sensor, camera_image
    client = None
    try:
        # اتصال و تنظیمات اولیه
        client = carla.Client('localhost', 2000)
        client.set_timeout(10.0)
        # بارگذاری نقشه جدید (مثلا Town03)
        world = client.get_world()
        
        settings = world.get_settings()
        settings.synchronous_mode = True 
        settings.fixed_delta_seconds = 0.1
        world.apply_settings(settings)
        
        # ایجاد خودرو و LiDAR (مانند قبل)
        vehicle_bp = world.get_blueprint_library().filter('vehicle.*')[0]
        spawn_point = world.get_map().get_spawn_points()[0]
        vehicle = world.spawn_actor(vehicle_bp, spawn_point)
        
        # تنظیم LiDAR
        lidar_bp = world.get_blueprint_library().find('sensor.lidar.ray_cast')
        # ... (تنظیمات LiDAR)
        lidar_bp.set_attribute('range', str(rmax))
        lidar_bp.set_attribute('rotation_frequency', '10') 
        lidar_bp.set_attribute('channels', '32') 
        lidar_bp.set_attribute('points_per_second', '64000') 

        lidar_transform = carla.Transform(carla.Location(x=0, z=2.5))
        lidar_sensor = world.spawn_actor(lidar_bp, lidar_transform, attach_to=vehicle)
        lidar_sensor.listen(lambda data: lidar_callback(data))

        # --- ایجاد سنسور دوربین RGB جدید ---
        camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
        # تنظیم ابعاد و فیلد دید برای دوربین
        camera_bp.set_attribute('image_size_x', '640')
        camera_bp.set_attribute('image_size_y', '480')
        camera_bp.set_attribute('fov', '100')
        
        # دوربین را روی سقف خودرو نصب کنید
        camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4)) 
        camera_sensor = world.spawn_actor(camera_bp, camera_transform, attach_to=vehicle)
        camera_sensor.listen(lambda data: camera_callback(data))
        print("✅ سنسور دوربین RGB ایجاد و متصل شد.")
        
        # فعال سازی Autopilot
        tm = client.get_trafficmanager(8000)  # پورت دلخواه Traffic Manager
        tm.set_synchronous_mode(True)
        vehicle.set_autopilot(True, tm.get_port())
        print("✅ شبیه‌سازی با حرکت Autopilot (سرعت پیش‌فرض) فعال شد.")
        
        # حلقه اصلی نقشه‌برداری
        cv2.namedWindow('Occupancy Grid Map', cv2.WINDOW_NORMAL)
        cv2.namedWindow('Vehicle Camera View', cv2.WINDOW_NORMAL) # پنجره جدید برای دوربین
        
        for t in range(3000):
            world.tick()
            
            # 1. دریافت وضعیت خودرو
            transform = vehicle.get_transform()
            x_carla = transform.location.x
            y_carla = transform.location.y
            theta_carla = math.radians(transform.rotation.yaw)
            
            # 2. جمع آوری اندازه‌گیری LiDAR
            meas_r = lidar_points_to_ranges(lidar_points, meas_phi, rmax)
            
            # 3. اجرای مدل سنسور معکوس
            # ... (محاسبه m_update و L_increment)
            m_update = inverse_scanner_carla(M, N, x_carla, y_carla, theta_carla, meas_phi, meas_r, rmax, alpha, beta)
            m_update_clipped = np.clip(m_update, 1e-6, 1 - 1e-6)
            L_increment = np.log(m_update_clipped / (1 - m_update_clipped)) - L0
            global L
            L = L + L_increment
            m = np.divide(np.exp(L), np.add(1, np.exp(L)))
            
            # --- نمایش نقشه (Occupancy Grid Map) ---
            map_display = np.array(m * 255, dtype=np.uint8)
            map_display = 255 - map_display 

            i_car, j_car = world_to_grid(x_carla, y_carla, M, N, GRID_RESOLUTION, GRID_ORIGIN_X, GRID_ORIGIN_Y)
            if i_car is not None and j_car is not None:
                cv2.circle(map_display, (j_car, i_car), 2, (0, 0, 255), -1) 
            
            cv2.imshow('Occupancy Grid Map', map_display)
            
            # --- نمایش نمای دوربین ---
            if camera_image is not None:
                # تبدیل از BGR (که OpenCV انتظار دارد) به RGB
                cv2.imshow('Vehicle Camera View', camera_image)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except Exception as e:
        print(f"❌ خطایی رخ داد: {e}")
        
    finally:
        # پاکسازی منابع
        print("💡 پاکسازی منابع CARLA...")
        if client:
            settings.synchronous_mode = False
            world.apply_settings(settings)
        if lidar_sensor:
            lidar_sensor.destroy()
        if camera_sensor: # سنسور دوربین را نیز پاک کنید
            camera_sensor.destroy()
        if vehicle:
            vehicle.destroy()
        cv2.destroyAllWindows()
        print("✅ شبیه‌سازی پایان یافت.")

if __name__ == '__main__':
    # این توابع باید قبل از اجرا در اسکریپت اصلی وجود داشته باشند
    # world_to_grid, grid_to_world, lidar_points_to_ranges, inverse_scanner_carla
    run_mapping_simulation()

✅ سنسور دوربین RGB ایجاد و متصل شد.
✅ شبیه‌سازی با حرکت Autopilot (سرعت پیش‌فرض) فعال شد.
💡 پاکسازی منابع CARLA...
✅ شبیه‌سازی پایان یافت.


KeyboardInterrupt: 

In [ ]:
# Ouput for grading. Do not modify this code!
m_f = ms[-1]

print("{}".format(m_f[40, 10]))
print("{}".format(m_f[30, 40]))
print("{}".format(m_f[35, 40]))
print("{}".format(m_f[0, 50]))
print("{}".format(m_f[10, 5]))
print("{}".format(m_f[20, 15]))
print("{}".format(m_f[25, 50]))  

Now that you have written your main simulation loop, you can visualize your robot motion in the true map, your measured belief map, and your occupancy grid belief map below. These are shown in the 1st, 2nd, and 3rd videos, respectively. If your 3rd video converges towards the true map shown in the 1st video, congratulations! You have completed the assignment. Please submit the output of the box above as a .txt file to the grader for this assignment.

In [ ]:
def map_update(i):
    map_ax.clear()
    map_ax.set_xlim(0, N)
    map_ax.set_ylim(0, M)
    map_ax.imshow(np.subtract(1, true_map), cmap='gray', origin='lower', vmin=0.0, vmax=1.0)
    x_plot = x[1, :i+1]
    y_plot = x[0, :i+1]
    map_ax.plot(x_plot, y_plot, "bx-")

def invmod_update(i):
    invmod_ax.clear()
    invmod_ax.set_xlim(0, N)
    invmod_ax.set_ylim(0, M)
    invmod_ax.imshow(invmods[i], cmap='gray', origin='lower', vmin=0.0, vmax=1.0)
    for j in range(len(meas_rs[i])):
        invmod_ax.plot(x[1, i] + meas_rs[i][j] * math.sin(meas_phi[j] + x[2, i]), \
                       x[0, i] + meas_rs[i][j] * math.cos(meas_phi[j] + x[2, i]), "ko")
    invmod_ax.plot(x[1, i], x[0, i], 'bx')
    
def belief_update(i):
    belief_ax.clear()
    belief_ax.set_xlim(0, N)
    belief_ax.set_ylim(0, M)
    belief_ax.imshow(ms[i], cmap='gray', origin='lower', vmin=0.0, vmax=1.0)
    belief_ax.plot(x[1, max(0, i-10):i], x[0, max(0, i-10):i], 'bx-')
    
map_anim = anim.FuncAnimation(map_fig, map_update, frames=len(x[0, :]), repeat=False)
invmod_anim = anim.FuncAnimation(invmod_fig, invmod_update, frames=len(x[0, :]), repeat=False)
belief_anim = anim.FuncAnimation(belief_fig, belief_update, frames=len(x[0, :]), repeat=False)


In [ ]:
from matplotlib.animation import HTMLWriter
HTML(map_anim.to_jshtml())


In [ ]:
HTML(invmod_anim.to_html5_video())

In [ ]:
HTML(belief_anim.to_html5_video())